## __JSON Metadata Parsing (DuckDB)__

Parses `data_cases.json` (the case-level API pull from `1-API_call.ipynb`) with DuckDB SQL into a flat, model-ready metadata table — restricted to the final tumor cohort, with one row per file. Originally prototyped in the DuckDB CLI before being moved into this notebook.

### __Imports__

In [23]:
import duckdb, os
import pandas as pd
from dotenv import load_dotenv
load_dotenv("project.env")

True

### __Load the UUID list__

In [10]:

UUID_list = pd.read_csv(os.path.join(os.getenv("Output_path"), "full_UUID_list.txt"))

duckdb.sql("""
    create or replace table UUID_list as 
    select *
    from UUID_list
""")

### __Build the primary metadata table__

Unnests cases, diagnoses, treatments and follow-ups from the raw JSON; computes age, 5-year survival, and collapsed cancer stage; restricted to file_ids in the tumor UUID list.

In [11]:
duckdb.sql("""
create or replace table primary_metadata as
with case_followup as (
    select
    case_id,
    max(followup.days_to_follow_up) as days_to_last_follow_up
    from (
        select
        hits.id as case_id,
        unnest(hits.follow_ups) as followup
        from (
            select unnest(data.hits) as hits
            from read_json('data_cases.json')
        )
    )
    group by case_id
)
select * from (
    select
    case_id,
    file_id,
    round(age_at_diagnosis / 365.0, 2) as age,
    race,
    gender,
    max(case
        when treatment_type = 'Surgery, NOS' then 1 else 0
    end) as Surgery_treatment,
    max(case
        when treatment_type like 'Radiation%' then 1 else 0
    end) as Radiation_treatment,
    max(case
        when treatment_type in (
            'Chemotherapy',
            'Hormone Therapy',
            'Immunotherapy (Including Vaccines)',
            'Targeted Molecular Therapy',
            'Pharmaceutical Therapy, NOS',
            'Bisphosphonate Therapy',
            'Ancillary Treatment'
        ) then 1 else 0
    end) as Pharmaceutical_treatment,
    max(case
        when vital_status = 'Dead'
        and days_to_death is not NULL
        and days_to_death < 5*365 then 0
        when vital_status = 'Dead'
        and days_to_death is not NULL
        and days_to_death >= 5*365 then 1
        when vital_status = 'Alive'
        and days_to_last_follow_up >= 5*365 then 1
        else NULL
    end) as survival_5yrs,
    case
        when cancer_stage in ('Stage I', 'Stage IA', 'Stage IB') then 1
        when cancer_stage in ('Stage II', 'Stage IIA', 'Stage IIB') then 2
        when cancer_stage in ('Stage III', 'Stage IIIA', 'Stage IIIB') then 3
        when cancer_stage in ('Stage IV') then 4
        else NULL
    end as cancer_stages_conca,
    from (
        select
        case_id,
        file_id,
        treatment.treatment_type as treatment_type,
        cf.days_to_last_follow_up as days_to_last_follow_up,
        age_at_diagnosis,
        cancer_stage,
        race,
        ethnicity,
        vital_status,
        days_to_death,
        gender,
        from (
            select
            case_id,
            file_id,
            unnest(treatments) as treatment,
            age_at_diagnosis,
            cancer_stage,
            race,
            ethnicity,
            vital_status,
            days_to_death,
            gender,
            from (
                select
                case_id,
                unnest(files).file_id as file_id,
                treatments,
                age_at_diagnosis,
                cancer_stage,
                race,
                ethnicity,
                vital_status,
                days_to_death,
                gender,
                from (
                    select
                    hits.id as case_id,
                    hits.files as files,
                    hits.demographic.race as race,
                    hits.demographic.ethnicity as ethnicity,
                    hits.demographic.vital_status as vital_status,
                    hits.demographic.days_to_death as days_to_death,
                    hits.demographic.sex_at_birth as gender,
                    unnest(hits.diagnoses).treatments as treatments,
                    unnest(hits.diagnoses).age_at_diagnosis as age_at_diagnosis,
                    unnest(hits.diagnoses).ajcc_pathologic_stage as cancer_stage,
                    from (
                        select unnest(data.hits) as hits
                        from read_json('data_cases.json')
                    )
                )
            )
        ) t
        left join case_followup cf using (case_id)
    )
    group by
    case_id,
    file_id,
    age_at_diagnosis,
    race,
    ethnicity,
    gender,
    vital_status,
    days_to_death,
    days_to_last_follow_up,
    cancer_stage
)
where file_id::varchar in (select UUID from UUID_list)""")

### __Aggregate to one row per file & export__

Collapses multiple diagnosis rows per file (`MAX`/`any_value`), drops files missing survival, stage, or age, and exports the final metadata table.

In [ ]:
duckdb.sql("""
CREATE OR REPLACE TABLE final_metadata AS
SELECT
    file_id,
    any_value(age) as age,
    any_value(race) as race,
    any_value(gender) as gender,
    MAX(Surgery_treatment) as Surgery_treatment,
    MAX(Radiation_treatment) as Radiation_treatment,
    MAX(Pharmaceutical_treatment) as Pharmaceutical_treatment,
    MAX(survival_5yrs) as survival_5yrs,
    MAX(cancer_stages_conca) as cancer_stages_conca
FROM primary_metadata
GROUP BY file_id
HAVING survival_5yrs IS NOT NULL
   AND cancer_stages_conca IS NOT NULL
   AND age IS NOT NULL
""")

metadata = duckdb.sql(""" select * from final_metadata""").df()
metadata.to_csv(os.path.join(os.getenv("Output_path"), "metadata.csv"), index=False)


Creating the list of final UUID for the parquet file creationg

In [ ]:
final_ids = duckdb.sql("select file_id from final_metadata").df()
final_ids["file_id"].to_csv(os.path.join(os.getenv("Output_path"),"selected_case_id.txt"), index=False, header=False)